# Narrador de Cenas com PyTorch + ONNX
### Classificação de ações (UCF-101 subset) → narração → app web

Neste projeto adaptamos o pipeline da aula (**CNN Residual → métricas → exportação ONNX**) para um **narrador de cenas**: o modelo lê frames de vídeo, reconhece a ação e gera um texto em português. A aplicação web (Streamlit) consome o `.onnx` e, no modo **monitoramento**, envia o aviso ao **Telegram**.

> Objetivo: treinar, avaliar, exportar ONNX e plugar em uma aplicação real.

---

## Tecnologias

**Python • PyTorch • Torchvision • ONNX • OpenCV • Streamlit • Telegram Bot API**

## Pipeline

**Dataset UCF-101 (10 ações) → Frames → Limpeza/EDA → Augmentation → CNN Residual → Treino → Teste → ONNX → Streamlit + Telegram**

## Por que UCF-101 (subset)?

- Padrão acadêmico de **reconhecimento de ações** (o que está acontecendo na cena)
- Subset público (~170 MB, 10 classes) viável para aula/Colab
- Classes narrativas claras (basquete, bebê engatinhando, banda marchando, etc.)
- Permanece alinhado à aula: classificação com CNN + export ONNX (sem VLM pesado)


## 1. Instalação e configuração


In [ ]:
!pip install -q imagehash onnx onnxscript onnxruntime opencv-python-headless huggingface_hub scikit-learn matplotlib pillow tqdm

In [ ]:
import os
import random
import warnings
import tarfile
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import imagehash

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

## 2. Download do dataset (UCF-101 subset)

Usamos o subset publicado em Hugging Face (`SayakPaul/ucf101-subset`): 10 classes, splits train/val/test.


In [ ]:
# Em Colab, descomente para gravar no Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = Path('/content/drive/MyDrive/Colab Notebooks/NarradorCenas')

PROJECT = Path('.').resolve()
DATA_DIR = PROJECT / 'data'
VIDEO_ROOT = DATA_DIR / 'UCF101_subset'
FRAMES_ROOT = DATA_DIR / 'frames'
MODELS_DIR = PROJECT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not VIDEO_ROOT.exists():
    from huggingface_hub import hf_hub_download
    archive = hf_hub_download(
        repo_id='SayakPaul/ucf101-subset',
        repo_type='dataset',
        filename='UCF101_subset.tar.gz',
        local_dir=str(DATA_DIR),
    )
    with tarfile.open(archive, 'r:*') as tar:
        tar.extractall(DATA_DIR)
    print('Dataset extraído.')
else:
    print('Dataset já presente.')

for split in ['train', 'val', 'test']:
    n = len(list((VIDEO_ROOT / split).rglob('*.avi')))
    print(f'{split}: {n} vídeos')
print('Classes:', sorted(p.name for p in (VIDEO_ROOT / 'train').iterdir() if p.is_dir()))

## 3. Extração de frames

O modelo da aula opera em **imagens**. Extraímos frames uniformemente de cada vídeo e organizamos no padrão `ImageFolder` (`split/classe/*.jpg`).


In [ ]:
def extrair_frames(video_root, frames_root, frames_por_video=6):
    for split in ['train', 'val', 'test']:
        split_in = video_root / split
        for class_dir in sorted(split_in.iterdir()):
            if not class_dir.is_dir():
                continue
            out_dir = frames_root / split / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            for video in sorted(class_dir.glob('*.avi')):
                cap = cv2.VideoCapture(str(video))
                total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
                if total <= 0:
                    cap.release()
                    continue
                idxs = np.linspace(0, total - 1, num=min(frames_por_video, total), dtype=int)
                for i, fi in enumerate(idxs):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
                    ok, frame = cap.read()
                    if ok:
                        cv2.imwrite(str(out_dir / f'{video.stem}_f{i:02d}.jpg'), frame)
                cap.release()

if not (FRAMES_ROOT / 'train').exists():
    print('Extraindo frames...')
    extrair_frames(VIDEO_ROOT, FRAMES_ROOT, frames_por_video=6)
else:
    print('Frames já extraídos.')

for split in ['train', 'val', 'test']:
    n = len(list((FRAMES_ROOT / split).rglob('*.jpg')))
    print(f'{split}: {n} frames')

## 4. Limpeza rápida e amostras


In [ ]:
def verificar_imagens(diretorio):
    corrompidas = []
    for root, _, files in os.walk(diretorio):
        for fname in files:
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            caminho = os.path.join(root, fname)
            try:
                with Image.open(caminho) as img:
                    img.verify()
                with Image.open(caminho) as img:
                    img.load()
            except Exception:
                corrompidas.append(caminho)
                os.remove(caminho)
    return corrompidas

for nome, pasta in [('train', FRAMES_ROOT/'train'), ('val', FRAMES_ROOT/'val'), ('test', FRAMES_ROOT/'test')]:
    rem = verificar_imagens(pasta)
    print(f'{nome}: {len(rem)} corrompida(s) removida(s)')

# Amostras
classes = sorted(p.name for p in (FRAMES_ROOT/'train').iterdir() if p.is_dir())
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cls in zip(axes.ravel(), classes):
    imgs = list((FRAMES_ROOT/'train'/cls).glob('*.jpg'))
    img = Image.open(random.choice(imgs))
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('Amostras — UCF-101 subset (frames)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Augmentation, DataLoaders e CNN Residual


In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

ds_train = ImageFolder(FRAMES_ROOT/'train', transform=transform_train)
ds_val = ImageFolder(FRAMES_ROOT/'val', transform=transform_eval)
ds_test = ImageFolder(FRAMES_ROOT/'test', transform=transform_eval)
loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False)
loader_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)
classes = ds_train.classes
print('Classes:', classes)
print(f'Treino={len(ds_train)} Val={len(ds_val)} Teste={len(ds_test)}')

In [ ]:
class BlocoResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bloco = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.projetor = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))
            if stride != 1 or in_ch != out_ch else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bloco(x) + self.projetor(x))


class CNNResidual(nn.Module):
    def __init__(self, n_classes=10, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1),
        )
        self.features = nn.Sequential(
            BlocoResidual(64, 64),
            BlocoResidual(64, 128, stride=2),
            BlocoResidual(128, 256, stride=2),
        )
        self.classificador = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classificador(self.features(self.stem(x)))


modelo = CNNResidual(n_classes=len(classes)).to(device)
print(f'Parâmetros: {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}')

## 6. Treinamento (AdamW, label smoothing, early stopping)


In [ ]:
LR = 3e-4
NUM_EPOCHS = 15
PACIENCIA = 5
otimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
criterio = nn.CrossEntropyLoss(label_smoothing=0.1)

def treinar_epoca(modelo, loader):
    modelo.train()
    loss_t = ok = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        otimizador.zero_grad()
        logits = modelo(x)
        loss = criterio(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        otimizador.step()
        loss_t += loss.item() * x.size(0)
        ok += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_t / n, 100 * ok / n

@torch.no_grad()
def avaliar(modelo, loader):
    modelo.eval()
    loss_t = ok = n = 0
    preds_all, labels_all = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = modelo(x)
        loss = criterio(logits, y)
        pred = logits.argmax(1)
        loss_t += loss.item() * x.size(0)
        ok += (pred == y).sum().item()
        n += x.size(0)
        preds_all.append(pred.cpu())
        labels_all.append(y.cpu())
    return loss_t / n, 100 * ok / n, torch.cat(preds_all).numpy(), torch.cat(labels_all).numpy()

historico = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
melhor = float('inf')
stale = 0
CKPT = MODELS_DIR / 'melhor_modelo.pth'

for epoca in range(1, NUM_EPOCHS + 1):
    tr_l, tr_a = treinar_epoca(modelo, loader_train)
    va_l, va_a, _, _ = avaliar(modelo, loader_val)
    historico['train_loss'].append(tr_l)
    historico['train_acc'].append(tr_a)
    historico['val_loss'].append(va_l)
    historico['val_acc'].append(va_a)
    mark = ''
    if va_l < melhor:
        melhor, stale = va_l, 0
        torch.save(modelo.state_dict(), CKPT)
        mark = '*'
    else:
        stale += 1
    print(f'Ep {epoca:02d} | T {tr_l:.4f}/{tr_a:.1f}% | V {va_l:.4f}/{va_a:.1f}% {mark}')
    if stale >= PACIENCIA:
        print('Early stopping')
        break

modelo.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
print('Melhor checkpoint restaurado.')

In [ ]:
epocas = range(1, len(historico['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epocas, historico['train_loss'], label='Treino')
axes[0].plot(epocas, historico['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epocas, historico['train_acc'], label='Treino')
axes[1].plot(epocas, historico['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Histórico de treinamento', fontweight='bold')
plt.tight_layout(); plt.show()

te_l, te_a, preds, labels = avaliar(modelo, loader_test)
print(f'Teste: loss={te_l:.4f} acc={te_a:.2f}%')
print(classification_report(labels, preds, target_names=classes, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(labels, preds)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
ax.set_title('Matriz de confusão — teste')
plt.tight_layout(); plt.show()

## 7. Narração a partir de um vídeo

Agrupamos previsões de frames amostrados e montamos um texto em português (templates por classe).


In [ ]:
NARRATION = {
    'ApplyEyeMakeup': 'Na cena, alguém está aplicando maquiagem nos olhos.',
    'ApplyLipstick': 'Na cena, alguém está passando batom.',
    'Archery': 'Na cena, uma pessoa está praticando arco e flecha.',
    'BabyCrawling': 'Na cena, um bebê está engatinhando.',
    'BalanceBeam': 'Na cena, uma ginasta está se equilibrando na trave.',
    'BandMarching': 'Na cena, uma banda está marchando em formação.',
    'BaseballPitch': 'Na cena, um jogador está arremessando no beisebol.',
    'Basketball': 'Na cena, alguém está jogando basquete.',
    'BasketballDunk': 'Na cena, um jogador está enterrando a bola no basquete.',
    'BenchPress': 'Na cena, alguém está fazendo exercício de supino.',
}

def narrar_video(caminho, modelo, classes, device, every_n=15, max_frames=30):
    cap = cv2.VideoCapture(str(caminho))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    preds = []
    idx = 0
    modelo.eval()
    while len(preds) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % every_n == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(rgb)
            tensor = transform_eval(img).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = F.softmax(modelo(tensor), dim=1).squeeze().cpu()
            i = int(probs.argmax())
            preds.append({
                't': idx / fps,
                'class': classes[i],
                'conf': float(probs[i]),
                'text': NARRATION.get(classes[i], classes[i]),
            })
        idx += 1
    cap.release()
    # dominante
    from collections import Counter
    cnt = Counter(p['class'] for p in preds if p['conf'] >= 0.35)
    if not cnt:
        return 'Cena indefinida.', preds
    dom = cnt.most_common(1)[0][0]
    avg = np.mean([p['conf'] for p in preds if p['class'] == dom])
    resumo = f"{NARRATION.get(dom, dom)} (confiança média: {avg:.0%})"
    return resumo, preds

# Demo com um vídeo de teste
demo = next((VIDEO_ROOT/'test').rglob('*.avi'))
resumo, detalhes = narrar_video(demo, modelo, classes, device)
print('Vídeo:', demo.name)
print('Narração:', resumo)
for d in detalhes[:8]:
    print(f"  {d['t']:.1f}s → {d['class']} ({d['conf']:.0%})")

## 8. Exportação ONNX (+ metadados)


In [ ]:
import json
import onnx

modelo.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
onnx_path = MODELS_DIR / 'narrador_cenas.onnx'

torch.onnx.export(
    modelo,
    dummy,
    str(onnx_path),
    input_names=['imagem'],
    output_names=['predicoes'],
    opset_version=18,
    dynamo=False,
)

m = onnx.load(str(onnx_path))
meta = {
    'task': 'action_recognition_frame',
    'classes': json.dumps(classes),
    'img_size': str(IMG_SIZE),
    'mean': json.dumps(MEAN),
    'std': json.dumps(STD),
    'color_mode': 'RGB',
    'framework': 'PyTorch',
    'architecture': 'CNNResidual',
    'dataset': 'UCF101_subset',
}
for k, v in meta.items():
    p = m.metadata_props.add()
    p.key, p.value = k, v
onnx.save(m, str(onnx_path))
print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1e6:.2f} MB)')

## 9. Aplicação web + Telegram

Com o `.onnx` gerado:

```bash
pip install -r requirements.txt
streamlit run app/streamlit_app.py
```

No app:
1. Envie um vídeo (ou use `sample_data/`)
2. Clique em **Narrar cena**
3. Ative **Monitoramento** e informe token + chat id do Telegram para receber o texto da cena

Variáveis: `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID`

---

### Resumo das decisões

| Decisão | Escolha | Motivo |
|---|---|---|
| Dataset | UCF-101 subset (10 ações) | Ideal para “o que acontece na cena”, tamanho de aula |
| Modelo | CNN Residual (aula) | Mesma base pedagógica + export ONNX simples |
| Narração | Templates PT-BR | Deployável sem LLM; texto claro para Telegram |
| App | Streamlit | Rápido para demo de upload de vídeo |
| Alertas | Telegram Bot API | Monitoramento sob demanda do usuário |
